# Decision Tree Classifiers for the Penguins Dataset

This notebook implements two Decision Tree classifiers on the Palmer Penguins dataset:
- **Classifier 1**: Predict **species** (Adelie, Chinstrap, Gentoo)
- **Classifier 2**: Predict **sex** (Male, Female)

Each classifier includes training, decision tree visualization, and full performance evaluation.

## 1. Imports and Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
)

# Suppress warnings for cleaner output
import warnings
warnings.filterwarnings("ignore")

# Reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("All libraries imported successfully.")

## 2. Load and Explore the Dataset

In [ ]:
# Load the Palmer Penguins dataset via seaborn (fetches from built-in cache)
df_raw = sns.load_dataset("penguins")

print(f"Dataset shape: {df_raw.shape}")
print(f"\nColumn dtypes:\n{df_raw.dtypes}")
print(f"\nMissing values per column:\n{df_raw.isnull().sum()}")
df_raw.head(10)

In [ ]:
print("Class distribution — Species:")
print(df_raw["species"].value_counts())
print("\nClass distribution — Sex:")
print(df_raw["sex"].value_counts())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

df_raw["species"].value_counts().plot(kind="bar", ax=axes[0], color=["steelblue", "tomato", "seagreen"], edgecolor="black")
axes[0].set_title("Species Distribution", fontsize=13)
axes[0].set_xlabel("Species")
axes[0].set_ylabel("Count")
axes[0].tick_params(axis="x", rotation=0)

df_raw["sex"].value_counts().plot(kind="bar", ax=axes[1], color=["orchid", "cornflowerblue"], edgecolor="black")
axes[1].set_title("Sex Distribution", fontsize=13)
axes[1].set_xlabel("Sex")
axes[1].set_ylabel("Count")
axes[1].tick_params(axis="x", rotation=0)

plt.tight_layout()
plt.show()

## 3. Data Preprocessing

In [ ]:
# Drop rows with any missing values (11 rows affected)
df = df_raw.dropna().reset_index(drop=True)
print(f"Rows after dropping NaNs: {len(df)}  (removed {len(df_raw) - len(df)} rows)")

# Encode categorical features: island and sex -> integer codes
le_island = LabelEncoder()
le_sex    = LabelEncoder()
le_species = LabelEncoder()

df["island_enc"] = le_island.fit_transform(df["island"])      # Biscoe=0, Dream=1, Torgersen=2
df["sex_enc"]    = le_sex.fit_transform(df["sex"])            # Female=0, Male=1
df["species_enc"]= le_species.fit_transform(df["species"])    # Adelie=0, Chinstrap=1, Gentoo=2

print(f"\nIsland encoding: {dict(zip(le_island.classes_, le_island.transform(le_island.classes_)))}")
print(f"Sex encoding:    {dict(zip(le_sex.classes_,    le_sex.transform(le_sex.classes_)))}")
print(f"Species encoding:{dict(zip(le_species.classes_,le_species.transform(le_species.classes_)))}")

# Feature set: all numeric measurements + encoded island
FEATURE_COLS = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g", "island_enc"]

print(f"\nFeatures used: {FEATURE_COLS}")
df[FEATURE_COLS].describe()

## 4. Classifier 1 — Predict Species

### 4.1 Train/Test Split and Model Training

In [ ]:
X = df[FEATURE_COLS]
y_species = df["species_enc"]

X_train_sp, X_test_sp, y_train_sp, y_test_sp = train_test_split(
    X, y_species, test_size=0.20, random_state=RANDOM_STATE, stratify=y_species
)

# Train decision tree — max_depth=4 keeps the visualization readable
dt_species = DecisionTreeClassifier(
    criterion="gini",
    max_depth=4,
    min_samples_leaf=3,
    random_state=RANDOM_STATE
)
dt_species.fit(X_train_sp, y_train_sp)

print(f"Training samples : {len(X_train_sp)}")
print(f"Test samples     : {len(X_test_sp)}")
print(f"Tree depth (fitted): {dt_species.get_depth()}")
print(f"Number of leaves   : {dt_species.get_n_leaves()}")

### 4.2 Decision Tree Visualization — Species

In [ ]:
SPECIES_NAMES = le_species.classes_.tolist()   # ['Adelie', 'Chinstrap', 'Gentoo']
FEATURE_NAMES = FEATURE_COLS

fig, ax = plt.subplots(figsize=(22, 10))
plot_tree(
    dt_species,
    feature_names=FEATURE_NAMES,
    class_names=SPECIES_NAMES,
    filled=True,
    rounded=True,
    fontsize=9,
    ax=ax,
    impurity=True,
    precision=3,
)
ax.set_title("Decision Tree — Species Classifier (max_depth=4)", fontsize=15, fontweight="bold")
plt.tight_layout()
plt.savefig("species_tree.png", dpi=150, bbox_inches="tight")
plt.show()

# Also print text representation for quick inspection
print(export_text(dt_species, feature_names=FEATURE_NAMES))

### 4.3 Feature Importances — Species

In [ ]:
importances_sp = pd.Series(dt_species.feature_importances_, index=FEATURE_NAMES).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8, 4))
importances_sp.plot(kind="barh", ax=ax, color="steelblue", edgecolor="black")
ax.set_title("Feature Importances — Species Classifier", fontsize=13)
ax.set_xlabel("Gini Importance")
for bar, val in zip(ax.patches, importances_sp):
    ax.text(val + 0.005, bar.get_y() + bar.get_height() / 2,
            f"{val:.3f}", va="center", fontsize=9)
plt.tight_layout()
plt.show()

### 4.4 Performance Evaluation — Species

In [ ]:
y_pred_sp = dt_species.predict(X_test_sp)

# --- Accuracy ---
train_acc_sp = accuracy_score(y_train_sp, dt_species.predict(X_train_sp))
test_acc_sp  = accuracy_score(y_test_sp, y_pred_sp)
print(f"Species Classifier")
print(f"  Train Accuracy : {train_acc_sp:.4f}")
print(f"  Test  Accuracy : {test_acc_sp:.4f}")

# --- Stratified 5-fold CV ---
cv_sp = cross_val_score(dt_species, X, y_species,
                         cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE),
                         scoring="accuracy")
print(f"  5-Fold CV Accuracy: {cv_sp.mean():.4f} ± {cv_sp.std():.4f}  (per fold: {np.round(cv_sp, 4)})")

# --- Classification Report ---
print(f"\nClassification Report (test set):\n")
print(classification_report(y_test_sp, y_pred_sp, target_names=SPECIES_NAMES))

# --- Confusion Matrix ---
cm_sp = confusion_matrix(y_test_sp, y_pred_sp)
fig, ax = plt.subplots(figsize=(6, 5))
disp = ConfusionMatrixDisplay(confusion_matrix=cm_sp, display_labels=SPECIES_NAMES)
disp.plot(ax=ax, colorbar=False, cmap="Blues")
ax.set_title("Confusion Matrix — Species Classifier", fontsize=13)
plt.tight_layout()
plt.show()

## 5. Classifier 2 — Predict Sex

### 5.1 Train/Test Split and Model Training

In [ ]:
# For sex classification we include the encoded species as a feature
# (sex is correlated with body measurements differently across species)
FEATURE_COLS_SEX = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g",
                    "island_enc", "species_enc"]

X_sex = df[FEATURE_COLS_SEX]
y_sex = df["sex_enc"]

X_train_sx, X_test_sx, y_train_sx, y_test_sx = train_test_split(
    X_sex, y_sex, test_size=0.20, random_state=RANDOM_STATE, stratify=y_sex
)

dt_sex = DecisionTreeClassifier(
    criterion="gini",
    max_depth=4,
    min_samples_leaf=3,
    random_state=RANDOM_STATE
)
dt_sex.fit(X_train_sx, y_train_sx)

print(f"Training samples : {len(X_train_sx)}")
print(f"Test samples     : {len(X_test_sx)}")
print(f"Tree depth (fitted): {dt_sex.get_depth()}")
print(f"Number of leaves   : {dt_sex.get_n_leaves()}")

### 5.2 Decision Tree Visualization — Sex

In [ ]:
SEX_NAMES = le_sex.classes_.tolist()   # ['Female', 'Male']

fig, ax = plt.subplots(figsize=(22, 10))
plot_tree(
    dt_sex,
    feature_names=FEATURE_COLS_SEX,
    class_names=SEX_NAMES,
    filled=True,
    rounded=True,
    fontsize=9,
    ax=ax,
    impurity=True,
    precision=3,
)
ax.set_title("Decision Tree — Sex Classifier (max_depth=4)", fontsize=15, fontweight="bold")
plt.tight_layout()
plt.savefig("sex_tree.png", dpi=150, bbox_inches="tight")
plt.show()

print(export_text(dt_sex, feature_names=FEATURE_COLS_SEX))

### 5.3 Feature Importances — Sex

In [ ]:
importances_sx = pd.Series(dt_sex.feature_importances_, index=FEATURE_COLS_SEX).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8, 4))
importances_sx.plot(kind="barh", ax=ax, color="orchid", edgecolor="black")
ax.set_title("Feature Importances — Sex Classifier", fontsize=13)
ax.set_xlabel("Gini Importance")
for bar, val in zip(ax.patches, importances_sx):
    ax.text(val + 0.005, bar.get_y() + bar.get_height() / 2,
            f"{val:.3f}", va="center", fontsize=9)
plt.tight_layout()
plt.show()

### 5.4 Performance Evaluation — Sex

In [ ]:
y_pred_sx = dt_sex.predict(X_test_sx)

train_acc_sx = accuracy_score(y_train_sx, dt_sex.predict(X_train_sx))
test_acc_sx  = accuracy_score(y_test_sx, y_pred_sx)
print(f"Sex Classifier")
print(f"  Train Accuracy : {train_acc_sx:.4f}")
print(f"  Test  Accuracy : {test_acc_sx:.4f}")

cv_sx = cross_val_score(dt_sex, X_sex, y_sex,
                         cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE),
                         scoring="accuracy")
print(f"  5-Fold CV Accuracy: {cv_sx.mean():.4f} ± {cv_sx.std():.4f}  (per fold: {np.round(cv_sx, 4)})")

print(f"\nClassification Report (test set):\n")
print(classification_report(y_test_sx, y_pred_sx, target_names=SEX_NAMES))

cm_sx = confusion_matrix(y_test_sx, y_pred_sx)
fig, ax = plt.subplots(figsize=(5, 4))
disp = ConfusionMatrixDisplay(confusion_matrix=cm_sx, display_labels=SEX_NAMES)
disp.plot(ax=ax, colorbar=False, cmap="Purples")
ax.set_title("Confusion Matrix — Sex Classifier", fontsize=13)
plt.tight_layout()
plt.show()

## 6. Side-by-Side Performance Summary

In [ ]:
summary = pd.DataFrame({
    "Classifier"       : ["Species (3-class)", "Sex (binary)"],
    "Train Accuracy"   : [f"{train_acc_sp:.4f}", f"{train_acc_sx:.4f}"],
    "Test Accuracy"    : [f"{test_acc_sp:.4f}",  f"{test_acc_sx:.4f}"],
    "CV Mean ± Std"    : [f"{cv_sp.mean():.4f} ± {cv_sp.std():.4f}",
                          f"{cv_sx.mean():.4f} ± {cv_sx.std():.4f}"],
    "Tree Depth"       : [dt_species.get_depth(), dt_sex.get_depth()],
    "Leaves"           : [dt_species.get_n_leaves(), dt_sex.get_n_leaves()],
}).set_index("Classifier")

print(summary.to_string())

# Bar chart comparison
fig, ax = plt.subplots(figsize=(8, 4))
metrics = ["Train Accuracy", "Test Accuracy", "CV Mean"]
sp_vals = [train_acc_sp, test_acc_sp, cv_sp.mean()]
sx_vals = [train_acc_sx, test_acc_sx, cv_sx.mean()]

x = np.arange(len(metrics))
width = 0.35
bars1 = ax.bar(x - width/2, sp_vals, width, label="Species", color="steelblue", edgecolor="black")
bars2 = ax.bar(x + width/2, sx_vals, width, label="Sex",     color="orchid",    edgecolor="black")

ax.set_ylim(0.7, 1.05)
ax.set_xticks(x)
ax.set_xticklabels(metrics)
ax.set_ylabel("Accuracy")
ax.set_title("Performance Comparison: Species vs. Sex Classifier", fontsize=13)
ax.legend()

for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
            f"{bar.get_height():.3f}", ha="center", fontsize=9)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
            f"{bar.get_height():.3f}", ha="center", fontsize=9)

plt.tight_layout()
plt.show()

## 7. Summary and Observations

### Species Classifier
- Uses features: bill length, bill depth, flipper length, body mass, island.
- Decision trees achieve near-perfect accuracy on species classification because the three penguin species occupy well-separated regions in morphological feature space. `flipper_length_mm` and `bill_length_mm` are typically the dominant splits.
- The 3-class confusion matrix reveals whether any Chinstrap/Adelie pairs are confused (they overlap the most in bill measurements).

### Sex Classifier
- Same morphological features plus `species_enc` as an additional feature, since sexual dimorphism varies by species.
- Sex prediction is a harder problem than species prediction: male and female penguins overlap more in their measurements. Body mass and bill depth are the most informative features.
- The binary confusion matrix shows any asymmetry in Female vs. Male errors.

### Key Takeaways
| Aspect | Species | Sex |
|---|---|---|
| Task difficulty | Easier (well-separated clusters) | Harder (overlapping distributions) |
| Primary features | flipper length, bill length | body mass, bill depth |
| Typical accuracy | ~97–100 % | ~88–92 % |
| Tree structure | Deeper splits needed for Chinstrap | Body mass threshold dominates root |